# Simuladores: Elección concurrente e Igualación
### Capítulo 14 — *Aprendizaje y Comportamiento Adaptable: Principios y Modelos*
**Arturo Bouzas** · Facultad de Psicología, UNAM · bouzaslab25.com

---
Ejecutar las celdas en orden.


---
**La ley generalizada de igualación (Baum, 1974)** describe la distribución de respuestas entre dos alternativas con la ecuación:



$$ \log(B_1/B_2) = \beta \cdot \log(r_1/r_2) + \log(\alpha) $$



**$\beta$ — Sensibilidad:** controla la pendiente de la función en escala logarítmica.
- Con $\beta = 1$ se obtiene igualación perfecta (las proporciones de respuesta igualan las de refuerzo).
- Con $\beta < 1$ hay *sub-igualación*: el organismo sub-discrimina las diferencias de refuerzo, distribuyendo sus respuestas de forma más uniforme de lo que predice la igualación perfecta.
- Con $\beta > 1$ hay *sobre-igualación*.

**$\log \alpha$ — Sesgo:** desplaza la función verticalmente sin cambiar su pendiente.
- Con $\log \alpha = 0$ (donde $\alpha = 1$) no hay sesgo. Valores positivos indican preferencia intrínseca por la opción 1, independiente de las tasas de refuerzo.

In [2]:
#@title **Simulador 14.1** — Ley Generalizada de Igualación
# ============================================================
# Simulador 14.1 — Ley Generalizada de Igualación
# Capítulo 14: Distribución del Comportamiento en Equilibrio
# Aprendizaje y Comportamiento Adaptable: Principios y Modelos
# ============================================================

# IMPORTACIONES
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output
import warnings
warnings.filterwarnings("ignore")

# ── Paletas claro / oscuro ────────────────────────────────────
_PALETAS_141 = {
    'claro': dict(
        azul       = '#2C5282',
        naranja    = '#C05621',
        verde      = '#276749',
        gris       = '#718096',
        rojo       = '#9B2335',
        fig_bg     = 'white',
        ax_bg      = 'white',
        texto      = '#2D3748',
        legend_bg  = 'white',
        panel_bg   = '#EBF4FF',
        panel_bord = '#2C5282',
        header_bg  = '#2C5282',
        header_fg  = 'white',
        sec_color  = '#2C5282',
    ),
    'oscuro': dict(
        azul       = '#90CDF4',
        naranja    = '#FBD38D',
        verde      = '#9AE6B4',
        gris       = '#A0AEC0',
        rojo       = '#FC8181',
        fig_bg     = '#1A202C',
        ax_bg      = '#2D3748',
        texto      = '#E2E8F0',
        legend_bg  = '#2D3748',
        panel_bg   = '#2D3748',
        panel_bord = '#4A5568',
        header_bg  = '#1A365D',
        header_fg  = '#EBF8FF',
        sec_color  = '#90CDF4',
    ),
}

def _p_141(oscuro: bool) -> dict:
    return _PALETAS_141['oscuro'] if oscuro else _PALETAS_141['claro']

# ── Lógica del modelo ─────────────────────────────────────────
def igualacion_proporcion_141(r_rel, alpha, beta):
    """
    Tasa relativa de respuesta según la ley generalizada de igualación.
    r_rel : proporción de refuerzo para la opción 1 (entre 0 y 1)
    alpha : parámetro de sesgo (> 0)
    beta  : parámetro de sensibilidad (> 0)
    Devuelve B1/(B1+B2).
    """
    eps = 1e-9
    r_ratio = np.clip(r_rel, eps, 1 - eps) / np.clip(1 - r_rel, eps, 1)
    b_ratio  = alpha * r_ratio ** beta
    return b_ratio / (1.0 + b_ratio)

# ── Visualización ────────────────────────────────────────────
def graficar_141(beta, log_alpha, mostrar_ref, mostrar_curvas, tema):

    oscuro = (tema == 'Oscuro')
    p = _p_141(oscuro)

    plt.rcParams.update({
        'font.family'       : 'serif',
        'figure.facecolor'  : p['fig_bg'],
        'axes.facecolor'    : p['ax_bg'],
        'axes.edgecolor'    : p['gris'],
        'axes.spines.top'   : False,
        'axes.spines.right' : False,
        'axes.grid'         : True,
        'grid.alpha'        : 0.30,
        'grid.color'        : p['gris'],
        'axes.labelcolor'   : p['gris'],
        'xtick.color'       : p['gris'],
        'ytick.color'       : p['gris'],
        'text.color'        : p['texto'],
        'legend.facecolor'  : p['legend_bg'],
        'legend.edgecolor'  : p['gris'],
        'legend.labelcolor' : p['texto'],
        'axes.labelsize'    : 11,
        'xtick.labelsize'   : 10,
        'ytick.labelsize'   : 10,
    })

    alpha    = 10.0 ** log_alpha
    r_rel    = np.linspace(0.001, 0.999, 600)
    b_rel    = igualacion_proporcion_141(r_rel, alpha, beta)

    log_r    = np.linspace(-2.2, 2.2, 600)
    log_b    = beta * log_r + log_alpha   # base 10: log₁₀(B1/B2) = β·log₁₀(r1/r2) + log₁₀(α)

    fig, (ax1, ax2) = plt.subplots(
        1, 2, figsize=(14, 6),
        gridspec_kw={'wspace': 0.25}
    )
    fig.patch.set_facecolor(p['fig_bg'])
    for ax in (ax1, ax2):
        ax.set_facecolor(p['ax_bg'])

    fig.suptitle('Ley Generalizada de Igualación', fontsize=14, fontweight='bold', color=p['texto'], y=1.01)

    # Panel izquierdo: Forma proporcional
    if mostrar_curvas:
        for b_ex, a_ex, lbl, col, ls in [
            (0.5, 1.0, 'Sub-igualación (β=0.5)', p['naranja'], '--'),
            (1.5, 1.0, 'Sobre-igualación (β=1.5)', p['verde'],   '-.'),
            (1.0, 2.0, 'Sesgo (α=2)',              p['rojo'],    ':'),
        ]:
            ax1.plot(r_rel, igualacion_proporcion_141(r_rel, a_ex, b_ex),
                     linestyle=ls, color=col, linewidth=1.4, alpha=0.7, label=lbl)

    if mostrar_ref:
        ax1.plot([0, 1], [0, 1], '--', color=p['gris'], linewidth=1.5,
                 label='Igualación perfecta (β=1, α=1)')

    ax1.plot(r_rel, b_rel, color=p['azul'], linewidth=2.8,
             label=f'Actual: β = {beta:.2f}, α = {alpha:.2f}')

    ax1.axhline(0.5, color=p['gris'], linewidth=0.7, linestyle=':', alpha=0.6)
    ax1.axvline(0.5, color=p['gris'], linewidth=0.7, linestyle=':', alpha=0.6)
    ax1.set_xlim(0, 1); ax1.set_ylim(0, 1)
    ax1.set_aspect('equal')
    ax1.set_xlabel(r'Tasa relativa de refuerzo $r_1\,/\,(r_1+r_2)$')
    ax1.set_ylabel(r'Tasa relativa de respuesta $B_1\,/\,(B_1+B_2)$')
    ax1.set_title('Forma proporcional', color=p['texto'], fontsize=12)
    ax1.legend(loc='upper left', frameon=True, framealpha=0.9)

    ax1.text(0.05, 0.93, 'Sobre-igualación', fontsize=8, color=p['gris'], transform=ax1.transAxes)
    ax1.text(0.55, 0.05, 'Sub-igualación', fontsize=8, color=p['gris'], transform=ax1.transAxes)

    # Panel derecho: Forma logarítmica
    if mostrar_curvas:
        for b_ex, a_ex, lbl, col, ls in [
            (0.5, 1.0, 'Sub-igualación (β=0.5)',   p['naranja'], '--'),
            (1.5, 1.0, 'Sobre-igualación (β=1.5)', p['verde'],   '-.'),
            (1.0, 2.0, 'Sesgo (log α=0.3)',        p['rojo'],    ':'),
        ]:
            log_b_ex = b_ex * log_r + np.log10(a_ex)
            ax2.plot(log_r, log_b_ex,
                     linestyle=ls, color=col, linewidth=1.4, alpha=0.7, label=lbl)

    if mostrar_ref:
        ax2.plot([-2.2, 2.2], [-2.2, 2.2], '--', color=p['gris'],
                 linewidth=1.5, label='Igualación perfecta')

    ax2.plot(log_r, log_b, color=p['azul'], linewidth=2.8,
             label=f'Actual: β = {beta:.2f}, log α = {log_alpha:.2f}')

    ax2.axhline(0, color=p['gris'], linewidth=0.7, linestyle=':', alpha=0.6)
    ax2.axvline(0, color=p['gris'], linewidth=0.7, linestyle=':', alpha=0.6)

    info = (f'Pendiente = β = {beta:.2f}\n'
            f'Intercepto = log α = {log_alpha:.2f}\n'
            f'(α = {alpha:.2f})')
    ax2.text(0.6, 0.15, info, transform=ax2.transAxes, fontsize=10,
             verticalalignment='top', color=p['texto'],
             bbox=dict(boxstyle='round,pad=0.5', facecolor=p['legend_bg'],
                       edgecolor=p['gris'], alpha=0.95))

    ax2.set_xlim(-2.2, 2.2); ax2.set_ylim(-3.0, 3.0)
    ax2.set_xlabel(r'$\log_{10}(r_1/r_2)$ — razón de refuerzo (escala log)')
    ax2.set_ylabel(r'$\log_{10}(B_1/B_2)$ — razón de respuesta (escala log)')
    ax2.set_title('Ley Generalizada — escala logarítmica', color=p['texto'], fontsize=12)
    ax2.legend(loc='upper left', frameon=True, framealpha=0.9)

    plt.show()

    # Instrucciones en Markdown (Se cambiaron a celda de texto)

def _html_header_141(oscuro: bool) -> str:
    p = _p_141(oscuro)
    return (
        f'<div style="'
        f'background-color:{p["header_bg"]};'
        f'color:{p["header_fg"]};'
        f'font-family:Georgia,serif;'
        f'font-size:14px;font-weight:bold;'
        f'padding:8px 14px;'
        f'border-radius:6px 6px 0 0;'
        f'letter-spacing:0.5px;">'
        f'&nbsp;Simulador 14.1 &mdash; Ley Generalizada de Igualación'
        f'</div>'
    )

def _html_sec_141(texto: str, oscuro: bool) -> str:
    p = _p_141(oscuro)
    return (
        f'<div style="'
        f'color:{p["sec_color"]};'
        f'font-family:Georgia,serif;'
        f'font-size:11px;font-weight:bold;'
        f'text-transform:uppercase;letter-spacing:1px;'
        f'margin:8px 0 2px 4px;">{texto}</div>'
    )

# ── Controles ─────────────────────────────────────────────────
estilo_141   = {'description_width': '210px'}
layout_l_141 = widgets.Layout(width='520px')

w_tema_141 = widgets.ToggleButtons(
    options=['Claro', 'Oscuro'],
    value='Claro',
    description='',
    style={'button_width': '120px'},
    layout=widgets.Layout(width='auto'),
)

w_beta_141 = widgets.FloatSlider(
    value=1.0, min=0.1, max=2.5, step=0.05,
    description='Sensibilidad β:',
    style=estilo_141, layout=layout_l_141,
    readout_format='.2f')

w_log_alpha_141 = widgets.FloatSlider(
    value=0.0, min=-0.9, max=0.9, step=0.05,
    description='Sesgo log₁₀(α):',
    style=estilo_141, layout=layout_l_141,
    readout_format='.2f')

w_ref_141 = widgets.Checkbox(
    value=True, description='Mostrar igualación perfecta',
    style=estilo_141, layout=layout_l_141)

w_curvas_141 = widgets.Checkbox(
    value=False, description='Mostrar ejemplos adicionales',
    style=estilo_141, layout=layout_l_141)

w_header_141 = widgets.HTML(value=_html_header_141(False))
w_sec1_141   = widgets.HTML(value=_html_sec_141('Tema', False))
w_sec2_141   = widgets.HTML(value=_html_sec_141('Parámetros de la Ley', False))
w_sec3_141   = widgets.HTML(value=_html_sec_141('Opciones de Visualización', False))

_body_layout_141 = widgets.Layout(
    padding='10px 16px 14px 16px',
    background_color=_PALETAS_141['claro']['panel_bg'],
    border=f'1px solid {_PALETAS_141["claro"]["panel_bord"]}',
    border_radius='0 0 6px 6px',
)

_body_141 = widgets.VBox(
    [
        w_sec1_141, w_tema_141,
        w_sec2_141,
        w_beta_141,
        w_log_alpha_141,
        w_sec3_141,
        w_ref_141,
        w_curvas_141,
    ],
    layout=_body_layout_141,
)

ui_141 = widgets.VBox([w_header_141, _body_141])

def _actualizar_tema_141(change):
    oscuro = (change['new'] == 'Oscuro')
    p      = _p_141(oscuro)

    w_header_141.value = _html_header_141(oscuro)
    w_sec1_141.value   = _html_sec_141('Tema', oscuro)
    w_sec2_141.value   = _html_sec_141('Parámetros de la Ley', oscuro)
    w_sec3_141.value   = _html_sec_141('Opciones de Visualización', oscuro)

    _body_141.layout.background_color = p['panel_bg']
    _body_141.layout.border           = f'1px solid {p["panel_bord"]}'

w_tema_141.observe(_actualizar_tema_141, names='value')

out_141 = widgets.interactive_output(
    graficar_141,
    {
        'beta'          : w_beta_141,
        'log_alpha'     : w_log_alpha_141,
        'mostrar_ref'   : w_ref_141,
        'mostrar_curvas': w_curvas_141,
        'tema'          : w_tema_141,
    }
)

display(ui_141, out_141)

Output()

---
**Ejercicios sugeridos:**
1. Ajusta $\beta = 0.7$ y $\log \alpha = 0$. Describe el tipo de desviación. ¿En qué punto de la curva proporcional la desviación respecto a igualación perfecta es mayor?
2. Con $\beta = 1.0$, ajusta $\log \alpha$ hasta que la curva pase por $B_1/(B_1+B_2) = 0.6$ cuando $r_1/(r_1+r_2) = 0.5$. ¿Qué valor de $\alpha$ produce ese resultado? ¿Qué significa?
3. Activa \"Mostrar ejemplos adicionales\" y compara las tres curvas. ¿Se puede distinguir sesgo de sensibilidad sin la escala logarítmica?

---
## Créditos y licencia

Este notebook es parte del proyecto:

> **Bouzas, A. (2026).** *Aprendizaje y Comportamiento Adaptable: Principios y Modelos.*
> Lab25, Facultad de Psicología, UNAM.
> https://www.bouzaslab25.com

Apoyo en la construcción del simulador: **Eduardo Sánchez**.

Código disponible en: **https://github.com/bouzaslab25/libro-aca**
Licencia: [CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)
